In [47]:
import numpy as np
import pandas as pd
from stocktrends import Renko
import statsmodels.api as sm
import yfinance as yf
import copy

def MACD(DF, a, b, c):
    df = DF.copy()
    df["MA_Fast"] = df["Close"].ewm(span=a, min_periods=a).mean()
    df["MA_Slow"] = df["Close"].ewm(span=b, min_periods=b).mean()
    df["MACD"] = df["MA_Fast"] - df["MA_Slow"]
    df["Signal"] = df["MACD"].ewm(span=c, min_periods=c).mean()
    return (df["MACD"], df["Signal"])

def ATR(DF, n):
    df = DF.copy()
    df['H-L'] = abs(df['High'] - df['Low'])
    df['H-PC'] = abs(df['High'] - df['Close'].shift(1))
    df['L-PC'] = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['H-L', 'H-PC', 'L-PC']].max(axis=1, skipna=False)
    df['ATR'] = df['TR'].rolling(n).mean()
    df2 = df.drop(['H-L', 'H-PC', 'L-PC'], axis=1)
    return df2

def slope(ser, n):
    slopes = [i * 0 for i in range(n - 1)]
    for i in range(n, len(ser) + 1):
        y = ser[i - n:i]
        x = np.array(range(n))
        y_scaled = (y - y.min()) / (y.max() - y.min())
        x_scaled = (x - x.min()) / (x.max() - x.min())
        x_scaled = sm.add_constant(x_scaled)
        model = sm.OLS(y_scaled, x_scaled)
        results = model.fit()
        slopes.append(results.params[-1])
    slope_angle = (np.rad2deg(np.arctan(np.array(slopes))))
    return np.array(slope_angle)

def renko_DF(DF):
    df = DF.copy()
    df.reset_index(inplace=True)
    df = df.loc[:, ["Date", "Open", "High", "Low", "Close", "Volume"]]
    df.columns = ["date", "open", "high", "low", "close", "volume"]
    df2 = Renko(df)
    df2.brick_size = max(0.5, round(ATR(DF, 120)["ATR"].dropna().iloc[-1], 0))
    renko_df = df2.get_ohlc_data()
    renko_df["bar_num"] = np.where(renko_df["uptrend"] == True, 1, np.where(renko_df["uptrend"] == False, -1, 0))
    for i in range(1, len(renko_df["bar_num"])):
        if renko_df["bar_num"].iloc[i] > 0 and renko_df["bar_num"].iloc[i - 1] > 0:
            renko_df.loc[i, "bar_num"] += renko_df["bar_num"].iloc[i - 1]
        elif renko_df["bar_num"].iloc[i] < 0 and renko_df["bar_num"].iloc[i - 1] < 0:
            renko_df.loc[i, "bar_num"] += renko_df["bar_num"].iloc[i - 1]
    renko_df.drop_duplicates(subset="date", keep="last", inplace=True)
    return renko_df

def CAGR(DF):
    df = DF.copy()
    df["cum_return"] = (1 + df["ret"]).cumprod()
    n = len(df) / (252)
    if n == 0 or len(df["cum_return"]) == 0:
        return 0.0
    CAGR = (df["cum_return"].tolist()[-1]) ** (1 / n) - 1
    return CAGR

def volatility(DF):
    df = DF.copy()
    vol = df["ret"].std() * np.sqrt(252)
    return vol if not np.isnan(vol) else 0.0

def sharpe(DF, rf):
    df = DF.copy()
    vol = volatility(df)
    if vol == 0:
        return -np.inf
    sr = (CAGR(df) - rf) / vol
    return sr

def max_dd(DF):
    df = DF.copy()
    df["cum_return"] = (1 + df["ret"]).cumprod()
    df["cum_roll_max"] = df["cum_return"].cummax()
    df["drawdown"] = df["cum_roll_max"] - df["cum_return"]
    df["drawdown_pct"] = df["drawdown"] / df["cum_roll_max"]
    max_dd = df["drawdown_pct"].max()
    return max_dd if not np.isnan(max_dd) else 0.0

tickers = ["MSFT", "AAPL", "AMZN", "INTC", "CSCO", "VZ", "IBM", "QCOM", "LYFT"]
ohlc_intraday = {}

for ticker in tickers:
    temp = yf.download(ticker, period="2y", interval="1d")
    temp.dropna(how="any", inplace=True)
    if isinstance(temp.columns, pd.MultiIndex):
        temp.columns = temp.columns.droplevel(1)
    ohlc_intraday[ticker] = temp

ohlc_renko = {}
df = copy.deepcopy(ohlc_intraday)
tickers_signal = {}
tickers_ret = {}

for ticker in tickers:
    renko = renko_DF(df[ticker])
    renko.columns = ["Date", "open", "high", "low", "close", "uptrend", "bar_num"]
    renko["Date"] = pd.to_datetime(renko["Date"])
    
    current_df = df[ticker].copy()
    current_df.index = pd.to_datetime(current_df.index)
    current_df.index.name = None
    
    ohlc_renko[ticker] = current_df.merge(renko.loc[:, ["Date", "bar_num"]], how="outer", left_index=True, right_on="Date").drop_duplicates(subset=["Date"], keep="last").set_index("Date")
    ohlc_renko[ticker]["bar_num"] = ohlc_renko[ticker]["bar_num"].ffill()
    
    macd_line, signal_line = MACD(ohlc_renko[ticker], 12, 26, 9)
    ohlc_renko[ticker]["macd"] = macd_line
    ohlc_renko[ticker]["macd_sig"] = signal_line
    
    ohlc_renko[ticker].dropna(subset=["macd", "macd_sig"], inplace=True)
    ohlc_renko[ticker].reset_index(drop=False, inplace=True)
    
    ohlc_renko[ticker]["macd_slope"] = slope(ohlc_renko[ticker]["macd"], 5)
    ohlc_renko[ticker]["macd_sig_slope"] = slope(ohlc_renko[ticker]["macd_sig"], 5)
    tickers_signal[ticker] = ""
    tickers_ret[ticker] = []

for ticker in tickers:
    for i in range(len(ohlc_renko[ticker])):
        if tickers_signal[ticker] == "":
            tickers_ret[ticker].append(0)
            if i > 0:
                if ohlc_renko[ticker]["bar_num"].iloc[i] >= 2 and ohlc_renko[ticker]["macd"].iloc[i] > ohlc_renko[ticker]["macd_sig"].iloc[i] and ohlc_renko[ticker]["macd_slope"].iloc[i] > ohlc_renko[ticker]["macd_sig_slope"].iloc[i]:
                    tickers_signal[ticker] = "Buy"
                elif ohlc_renko[ticker]["bar_num"].iloc[i] <= -2 and ohlc_renko[ticker]["macd"].iloc[i] < ohlc_renko[ticker]["macd_sig"].iloc[i] and ohlc_renko[ticker]["macd_slope"].iloc[i] < ohlc_renko[ticker]["macd_sig_slope"].iloc[i]:
                    tickers_signal[ticker] = "Sell"
        
        elif tickers_signal[ticker] == "Buy":
            tickers_ret[ticker].append((ohlc_renko[ticker]["Close"].iloc[i] / ohlc_renko[ticker]["Close"].iloc[i - 1]) - 1)
            if i > 0:
                if ohlc_renko[ticker]["bar_num"].iloc[i] <= -2 and ohlc_renko[ticker]["macd"].iloc[i] < ohlc_renko[ticker]["macd_sig"].iloc[i] and ohlc_renko[ticker]["macd_slope"].iloc[i] < ohlc_renko[ticker]["macd_sig_slope"].iloc[i]:
                    tickers_signal[ticker] = "Sell"
                elif ohlc_renko[ticker]["macd"].iloc[i] < ohlc_renko[ticker]["macd_sig"].iloc[i] and ohlc_renko[ticker]["macd_slope"].iloc[i] < ohlc_renko[ticker]["macd_sig_slope"].iloc[i]:
                    tickers_signal[ticker] = ""
                
        elif tickers_signal[ticker] == "Sell":
            tickers_ret[ticker].append((ohlc_renko[ticker]["Close"].iloc[i - 1] / ohlc_renko[ticker]["Close"].iloc[i]) - 1)
            if i > 0:
                if ohlc_renko[ticker]["bar_num"].iloc[i] >= 2 and ohlc_renko[ticker]["macd"].iloc[i] > ohlc_renko[ticker]["macd_sig"].iloc[i] and ohlc_renko[ticker]["macd_slope"].iloc[i] > ohlc_renko[ticker]["macd_sig_slope"].iloc[i]:
                    tickers_signal[ticker] = "Buy"
                elif ohlc_renko[ticker]["macd"].iloc[i] > ohlc_renko[ticker]["macd_sig"].iloc[i] and ohlc_renko[ticker]["macd_slope"].iloc[i] > ohlc_renko[ticker]["macd_sig_slope"].iloc[i]:
                    tickers_signal[ticker] = ""
                    
    ohlc_renko[ticker]["ret"] = np.array(tickers_ret[ticker])

strategy_df = pd.DataFrame()
for ticker in tickers:
    strategy_df[ticker] = ohlc_renko[ticker]["ret"]
strategy_df["ret"] = strategy_df.mean(axis=1)

print("Strategy CAGR:", CAGR(strategy_df))
print("Strategy Sharpe:", sharpe(strategy_df, 0.025))
print("Strategy Max DD:", max_dd(strategy_df))

cagr = {}
sharpe_ratios = {}
max_drawdown = {}
for ticker in tickers:
    cagr[ticker] = CAGR(ohlc_renko[ticker])
    sharpe_ratios[ticker] = sharpe(ohlc_renko[ticker], 0.025)
    max_drawdown[ticker] = max_dd(ohlc_renko[ticker])

KPI_df = pd.DataFrame([cagr, sharpe_ratios, max_drawdown], index=["Return", "Sharpe Ratio", "Max Drawdown"])
print(KPI_df.T)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
C:\Users\torre\AppData\Local\Temp\ipykernel_7584\551329547.py:36: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slopes.append(results.params[-1])
C:\Users\torre\AppData\Local\Temp\ipykernel_7

Strategy CAGR: 0.08015869224065608
Strategy Sharpe: 0.4527375200586003
Strategy Max DD: 0.07933687359606269
        Return  Sharpe Ratio  Max Drawdown
MSFT -0.044314     -0.380121      0.199292
AAPL  0.042192      0.081456      0.188594
AMZN  0.089335      0.288572      0.197115
INTC  0.102786      0.179625      0.374914
CSCO  0.170637      0.619273      0.188008
VZ   -0.022392     -0.404860      0.158591
IBM  -0.064891     -0.299525      0.408503
QCOM  0.020970     -0.011820      0.412140
LYFT  0.138368      0.298016      0.247035
